## Dynamic Batching Engine	
### Concurrency & Throughput

Implement a "waiting room" that gathers incoming inference requests 
and groups them into batches of size N or after T milliseconds to 
maximize GPU saturation.

```python
class DynamicBatcher:
    def __init__(self, max_batch_size: int, max_wait_time_ms: float):
        # Initialize your queues and state here
        pass

    async def generate(self, input_text: str) -> str:
        # 1. Wrap the input in a Request object
        # 2. Add it to the batching queue
        # 3. Wait for the result and return it
        pass

    async def _batch_loop(self):
        # The background logic that pulls requests, 
        # calls the mock model, and dispatches results.
        pass
```

### High-level design

**Goal:** Many clients call `generate()` at unpredictable times. Running one tiny batch per call wastes GPU parallelism. **Dynamic batching** waits (up to `max_wait_time_ms`) and/or collects up to `max_batch_size` requests, then runs **one** forward pass and splits outputs back to callers.

**Main pieces**

1. **`asyncio.Queue`** — Thread-safe async queue. Producers (`generate`) `put` requests; a single consumer (`_batch_loop`) `get`s them. This decouples arrival rate from batch formation.

2. **Per-request `Future`** — Each caller needs its own result. We attach an `asyncio.Future` to each `Request`. The batch loop completes each future with the matching output. `generate()` does `return await fut`, so callers naturally wait without blocking the event loop.

3. **Batching rule** — After the **first** request in a batch arrives, set a **deadline** = now + max wait. Keep pulling more requests with `asyncio.wait_for(queue.get(), timeout=remaining)` until either the batch is full or the deadline passes (or both). This implements **“N items OR T ms since first item”** (common server semantics).

4. **`time.monotonic()`** — Used for deadlines so clock adjustments (NTP) do not shrink or stretch wait windows.

5. **Background task** — `_batch_loop` runs forever as a task. Starting it from `__aenter__` guarantees a running event loop (unlike `create_task` in `__init__`).

**Why not `Queue.empty()`?** It is explicitly unreliable for coordination under concurrency; blocking `get` with a timeout encodes the wait window correctly.

**Why async mock inference?** A real GPU call is often synchronous from Python’s perspective; you would offload with `loop.run_in_executor` and a **plain `def`**. Here we use `asyncio.sleep` to keep the example single-threaded and easy to read.

In [ ]:
import asyncio
import time
from dataclasses import dataclass

@dataclass
class Request:
    input_text: str
    future: asyncio.Future[str]
    arrival_time: float  # wall time; useful for metrics / debugging


class DynamicBatcher:
    def __init__(self, max_batch_size: int, max_wait_time_ms: float):
        self.max_batch_size = max_batch_size
        self.max_wait_s = max_wait_time_ms / 1000.0
        self._queue = asyncio.Queue()
        self._batch_task = None

    async def __aenter__(self) -> "DynamicBatcher":
        self._batch_task = asyncio.create_task(self._batch_loop())

    async def __aexit__(self, exc_type, exc, tb) -> None:
        if self._batch_task is not None:
            self._batch_task.cancel()
            try:
                await self._batch_task
            except asyncio.CancelledError:
                pass

    async def generate(self, input_text: str) -> str:
        loop = asyncio.get_running_loop()
        fut = loop.create_future()
        req = Request(
            input_text=input_text, 
            future=fut, 
            arrival_time=time.time()
        )
        await self._queue.put(req)
        
        return await fut

    async def _mock_model_inference(self, texts: list[str]) -> list[str]:
        # Simulates async-friendly GPU pipeline; replace with real model call.
        await asyncio.sleep(0.05)
        return [f"processed: {t}" for t in texts]

    async def _batch_loop(self) -> None:
        while True:
            first = await self._queue.get()
            batch = [first]
            deadline = time.monotonic() + self.max_wait_s

            # Fill until max_batch_size OR deadline (whichever comes first)
            while len(batch) < self.max_batch_size:
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                try:
                    nxt = await asyncio.wait_for(self._queue.get(), timeout=remaining)
                    batch.append(nxt)
                except asyncio.TimeoutError:
                    break

            texts = [r.input_text for r in batch]
            try:
                outputs = await self._mock_model_inference(texts)
            except Exception as e:
                for r in batch:
                    if not r.future.done():
                        r.future.set_exception(e)
                continue

            for r, out in zip(batch, outputs, strict=True):
                if not r.future.done():
                    r.future.set_result(out)
